# Experiments – Edouard

Each experiment is a DLC shuffle that holds out one session (train on the 7 others).
Needs `dlc/pupil-dani-2026-11-13/` with its `labeled-data/`. Steps already done are skipped, so "Run All" is safe.

In [1]:
from pathlib import Path

import deeplabcut
import pandas as pd
from deeplabcut.utils import auxiliaryfunctions

PROJECT = Path("dlc") / "pupil-dani-2026-11-13"
MODELS_DIR = PROJECT / "dlc-models-pytorch" / "iteration-0"
EVAL_DIR = PROJECT / "evaluation-results-pytorch" / "iteration-0"
config_path = str(PROJECT / "config.yaml")

cfg = auxiliaryfunctions.read_config(config_path)
cfg["project_path"] = str(PROJECT.resolve())
auxiliaryfunctions.write_config(config_path, cfg)

merged = pd.read_hdf(next((PROJECT / auxiliaryfunctions.get_training_set_folder(cfg)).glob("CollectedData_*.h5")))
frame_sessions = [str(idx[1]) for idx in merged.index]
print(pd.Series(frame_sessions).value_counts().sort_index().to_string())  # 393 labeled frames over 8 sessions


def split_indices(test_session: str) -> tuple[list[int], list[int]]:
    """Return the (train, test) row indices of the merged labels, holding out one session."""
    test = [i for i, session in enumerate(frame_sessions) if session == test_session]
    train = [i for i, session in enumerate(frame_sessions) if session != test_session]
    if not test:
        raise ValueError(f"No labeled frames for session {test_session}.")
    return train, test


def shuffle_dir(shuffle: int) -> Path | None:
    """Return a shuffle's model folder, or None if the shuffle doesn't exist yet."""
    matches = list(MODELS_DIR.glob(f"*shuffle{shuffle}"))
    return matches[0] if matches else None


def create_shuffle(shuffle: int, test_session: str) -> None:
    """Create a shuffle holding out one session, unless the shuffle already exists."""
    if shuffle_dir(shuffle) is not None:
        print(f"Shuffle {shuffle} already exists, skipped.")
        return
    train, test = split_indices(test_session)
    print(f"Shuffle {shuffle}: test on {test_session} | train frames: {len(train)} | test frames: {len(test)}")
    # DLC names a shuffle after its train fraction and requires that fraction in config.yaml's TrainingFraction list.
    fraction = round(len(train) / (len(train) + len(test)), 2)
    project_cfg = auxiliaryfunctions.read_config(config_path)
    if fraction not in project_cfg["TrainingFraction"]:
        project_cfg["TrainingFraction"].append(fraction)
        auxiliaryfunctions.write_config(config_path, project_cfg)
    deeplabcut.create_training_dataset(
        config_path, Shuffles=[shuffle],
        trainIndices=[train], testIndices=[test],
        net_type="resnet_50",
    )


def trainingset_index(shuffle: int) -> int:
    """Position of a shuffle's train fraction in config.yaml's TrainingFraction list, as DLC's functions expect."""
    folder = shuffle_dir(shuffle)
    assert folder is not None, f"Shuffle {shuffle} doesn't exist."
    percent = int(folder.name.split("trainset")[1].split("shuffle")[0])
    fractions = auxiliaryfunctions.read_config(config_path)["TrainingFraction"]
    return [round(100 * f) for f in fractions].index(percent)


def configure_shuffle(shuffle: int, train: dict, inference: dict, epochs: int) -> None:
    """Overwrite chosen training settings in a shuffle's pytorch_config.yaml."""
    folder = shuffle_dir(shuffle)
    assert folder is not None, f"Shuffle {shuffle} doesn't exist."
    path = folder / "train" / "pytorch_config.yaml"
    pcfg = auxiliaryfunctions.read_plainconfig(path)
    pcfg["data"]["train"].update(train)
    pcfg["data"]["inference"].update(inference)
    pcfg["train_settings"]["epochs"] = epochs
    auxiliaryfunctions.write_plainconfig(path, pcfg)
    print(f"Shuffle {shuffle} | train: {train} | inference: {inference} | epochs: {epochs}")


def train_shuffle(shuffle: int, epochs: int) -> None:
    """Train a shuffle, unless its final snapshot already exists."""
    folder = shuffle_dir(shuffle)
    assert folder is not None, f"Shuffle {shuffle} doesn't exist."
    if (folder / "train" / f"snapshot-{epochs:03d}.pt").exists():
        print(f"Shuffle {shuffle} already trained, skipped.")
        return
    deeplabcut.train_network(config_path, shuffle=shuffle, trainingsetindex=trainingset_index(shuffle))


def evaluate_shuffle(shuffle: int, epochs: int, force: bool = False) -> None:
    """Evaluate a shuffle's final snapshot, unless already done (never the 'best' one, picked with the test score).

    hist_eq is random at test time in DLC, so re-evaluating can change the numbers: use force=True on purpose only.
    """
    done = list(EVAL_DIR.glob(f"*shuffle{shuffle}/*_snapshot_{epochs:03d}-results.csv"))
    if done and not force:
        print(f"Shuffle {shuffle} already evaluated, skipped.")
        return
    deeplabcut.evaluate_network(
        config_path, Shuffles=[shuffle], trainingsetindex=trainingset_index(shuffle),
        snapshots_to_evaluate=[f"snapshot-{epochs:03d}"],
        per_keypoint_evaluation=True,
    )

094124    40
110535    39
115300    34
131707    60
135059    40
171101    20
172731    80
214411    80


## Shuffle 2 – histogram equalization

`hist_eq` in train and inference (DLC applies it to each image with probability 0.5), 100 epochs.
Result on 110535: 93 px mean error (baseline 238 px), 7 % detection (baseline 0 %).

In [6]:
create_shuffle(2, test_session="110535")
configure_shuffle(2, train={"hist_eq": True}, inference={"hist_eq": True}, epochs=100)
train_shuffle(2, epochs=100)
evaluate_shuffle(2, epochs=100)

Shuffle 2 already exists, skipped.
Shuffle 2 | train: {'hist_eq': True} | inference: {'hist_eq': True} | epochs: 100
Shuffle 2 already trained, skipped.
Shuffle 2 already evaluated, skipped.


## Shuffle 3 – histogram equalization + horizontal flip

Same as shuffle 2, plus `hflip: 0.5` in training only (no keypoint swap: labels are anatomical).
Result on 110535: 21 px mean error, 36 % detection; the anterior/posterior swap is gone.

In [7]:
create_shuffle(3, test_session="110535")
configure_shuffle(3, train={"hist_eq": True, "hflip": 0.5}, inference={"hist_eq": True}, epochs=100)
train_shuffle(3, epochs=100)
evaluate_shuffle(3, epochs=100)

Shuffle 3 already exists, skipped.
Shuffle 3 | train: {'hist_eq': True, 'hflip': 0.5} | inference: {'hist_eq': True} | epochs: 100
Shuffle 3 already trained, skipped.
Shuffle 3 already evaluated, skipped.


## Development rounds – 135059 and 171101 held out

Does shuffle 3's gain hold on other sessions? Each round compares DLC's defaults with `hist_eq` + flip, both at 100 epochs.

| Shuffle | Held out | Settings |
|---|---|---|
| 4 | 135059 | DLC defaults |
| 5 | 135059 | hist_eq + hflip |
| 6 | 171101 | DLC defaults |
| 7 | 171101 | hist_eq + hflip |

About 2 h 30 of training in total.

In [8]:
BASELINE: dict[str, dict] = {"train": {}, "inference": {}}
HISTEQ_FLIP: dict[str, dict] = {"train": {"hist_eq": True, "hflip": 0.5}, "inference": {"hist_eq": True}}

DEV_RUNS: dict[int, tuple[str, dict[str, dict]]] = {
    4: ("135059", BASELINE),
    5: ("135059", HISTEQ_FLIP),
    6: ("171101", BASELINE),
    7: ("171101", HISTEQ_FLIP),
}

for shuffle, (test_session, settings) in DEV_RUNS.items():
    create_shuffle(shuffle, test_session=test_session)
    configure_shuffle(shuffle, train=settings["train"], inference=settings["inference"], epochs=100)

Shuffle 4 already exists, skipped.
Shuffle 4 | train: {} | inference: {} | epochs: 100
Shuffle 5 already exists, skipped.
Shuffle 5 | train: {'hist_eq': True, 'hflip': 0.5} | inference: {'hist_eq': True} | epochs: 100
Shuffle 6: test on 171101 | train frames: 373 | test frames: 20
You passed a split with the following fraction: 95%
Shuffle 6 | train: {} | inference: {} | epochs: 100
Shuffle 7: test on 171101 | train frames: 373 | test frames: 20
You passed a split with the following fraction: 95%
Shuffle 7 | train: {'hist_eq': True, 'hflip': 0.5} | inference: {'hist_eq': True} | epochs: 100


In [9]:
for shuffle in DEV_RUNS:
    train_shuffle(shuffle, epochs=100)

Training with configuration:
config_version: 0
model: backbone={'type': 'ResNet', 'model_name': 'resnet50_gn', 'output_stride': 16, 'freeze_bn_stats': False, 'freeze_bn_weights': False} heads={'bodypart': {'type': 'HeatmapHead', 'weight_init': 'normal', 'predictor': {'type': 'HeatmapPredictor', 'apply_sigmoid': False, 'clip_scores': True, 'location_refinement': True, 'locref_std': 7.2801}, 'target_generator': {'type': 'HeatmapGaussianGenerator', 'num_heatmaps': 8, 'pos_dist_thresh': 17, 'heatmap_mode': 'KEYPOINT', 'gradient_masking': False, 'generate_locref': True, 'locref_std': 7.2801}, 'criterion': {'heatmap': {'type': 'WeightedMSECriterion', 'weight': 1.0}, 'locref': {'type': 'WeightedHuberCriterion', 'weight': 0.05}}, 'heatmap_config': {'channels': [2048, 8], 'kernel_size': [3], 'strides': [2]}, 'locref_config': {'channels': [2048, 16], 'kernel_size': [3], 'strides': [2]}}} backbone_output_channels=2048 neck=None pose_model=None
net_type: resnet_50
method: bu
device: auto
metadata:

In [10]:
for shuffle in DEV_RUNS:
    evaluate_shuffle(shuffle, epochs=100)

Evaluation scorer: DLC_Resnet50_pupilNov13shuffle4_snapshot_100


100%|██████████| 40/40 [00:01<00:00, 20.75it/s]


Evaluation results file: DLC_Resnet50_pupilNov13shuffle4_snapshot_100-results.csv
Evaluation results for DLC_Resnet50_pupilNov13shuffle4_snapshot_100-results.csv (pcutoff: 0.6):
train rmse             2.67
train rmse_pcutoff     2.67
train mAP             98.66
train mAR             99.15
test rmse             61.53
test rmse_pcutoff     63.86
test mAP               2.33
test mAR               4.00
Name: (0.9, 4, 100, -1, 0.6), dtype: float64
Per-bodypart evaluation results (DLC_Resnet50_pupilNov13shuffle4_snapshot_100-keypoint-results):
  Train error (px)
    eye-anterior:      3.32px
    eye-posterior:     2.50px
    eye-dorsal:        3.14px
    eye-ventral:       3.35px
    pupil-anterior:    2.18px
    pupil-posterior:   2.42px
    pupil-dorsal:      2.24px
    pupil-ventral:     2.17px
  Test error (px)
    eye-anterior:      166.98px
    eye-posterior:     160.91px
    eye-dorsal:        13.54px
    eye-ventral:       7.19px
    pupil-anterior:    68.47px
    pupil-posterior:   

c:\Users\edoua\miniconda3\envs\ml4science\lib\site-packages\deeplabcut\pose_estimation_pytorch\config\data.py:170: UserWarning: Be careful! Do not train pose models with horizontal flips if you have symmetric keypoints!
  build_transforms(v)
100%|██████████| 40/40 [00:01<00:00, 22.32it/s]


Evaluation results file: DLC_Resnet50_pupilNov13shuffle5_snapshot_100-results.csv
Evaluation results for DLC_Resnet50_pupilNov13shuffle5_snapshot_100-results.csv (pcutoff: 0.6):
train rmse             3.44
train rmse_pcutoff     3.04
train mAP             97.76
train mAR             98.41
test rmse             60.23
test rmse_pcutoff     58.01
test mAP               5.03
test mAR               6.25
Name: (0.9, 5, 100, -1, 0.6), dtype: float64
Per-bodypart evaluation results (DLC_Resnet50_pupilNov13shuffle5_snapshot_100-keypoint-results):
  Train error (px)
    eye-anterior:      4.67px
    eye-posterior:     3.73px
    eye-dorsal:        4.60px
    eye-ventral:       4.74px
    pupil-anterior:    2.45px
    pupil-posterior:   2.61px
    pupil-dorsal:      2.35px
    pupil-ventral:     2.35px
  Test error (px)
    eye-anterior:      164.23px
    eye-posterior:     158.82px
    eye-dorsal:        9.00px
    eye-ventral:       7.18px
    pupil-anterior:    68.74px
    pupil-posterior:   6

100%|██████████| 20/20 [00:00<00:00, 23.86it/s]


Evaluation results file: DLC_Resnet50_pupilNov13shuffle6_snapshot_100-results.csv
Evaluation results for DLC_Resnet50_pupilNov13shuffle6_snapshot_100-results.csv (pcutoff: 0.6):
train rmse             2.67
train rmse_pcutoff     2.67
train mAP             98.60
train mAR             99.28
test rmse             15.26
test rmse_pcutoff      7.46
test mAP              75.55
test mAR              80.50
Name: (0.95, 6, 100, -1, 0.6), dtype: float64
Per-bodypart evaluation results (DLC_Resnet50_pupilNov13shuffle6_snapshot_100-keypoint-results):
  Train error (px)
    eye-anterior:      3.34px
    eye-posterior:     2.82px
    eye-dorsal:        3.14px
    eye-ventral:       3.29px
    pupil-anterior:    2.19px
    pupil-posterior:   2.41px
    pupil-dorsal:      2.10px
    pupil-ventral:     2.07px
  Test error (px)
    eye-anterior:      27.33px
    eye-posterior:     65.40px
    eye-dorsal:        5.13px
    eye-ventral:       4.70px
    pupil-anterior:    6.62px
    pupil-posterior:   6.8

c:\Users\edoua\miniconda3\envs\ml4science\lib\site-packages\deeplabcut\pose_estimation_pytorch\config\data.py:170: UserWarning: Be careful! Do not train pose models with horizontal flips if you have symmetric keypoints!
  build_transforms(v)
100%|██████████| 20/20 [00:00<00:00, 24.44it/s]


Evaluation results file: DLC_Resnet50_pupilNov13shuffle7_snapshot_100-results.csv
Evaluation results for DLC_Resnet50_pupilNov13shuffle7_snapshot_100-results.csv (pcutoff: 0.6):
train rmse             3.10
train rmse_pcutoff     2.96
train mAP             97.86
train mAR             98.69
test rmse              9.56
test rmse_pcutoff      4.75
test mAP              92.36
test mAR              94.50
Name: (0.95, 7, 100, -1, 0.6), dtype: float64
Per-bodypart evaluation results (DLC_Resnet50_pupilNov13shuffle7_snapshot_100-keypoint-results):
  Train error (px)
    eye-anterior:      4.73px
    eye-posterior:     3.15px
    eye-dorsal:        3.62px
    eye-ventral:       3.85px
    pupil-anterior:    2.38px
    pupil-posterior:   2.58px
    pupil-dorsal:      2.28px
    pupil-ventral:     2.20px
  Test error (px)
    eye-anterior:      45.86px
    eye-posterior:     8.87px
    eye-dorsal:        4.92px
    eye-ventral:       2.68px
    pupil-anterior:    2.42px
    pupil-posterior:   5.98

## Comparison of all experiments

Scores computed by `src/metrics.py` from the predictions DLC saved during evaluation (no retraining).

In [2]:
from src.data import load_labels
from src.metrics import compare_runs, evaluate_run, find_predictions, load_predictions

labels = load_labels(Path("data") / "labeled-data")

EXPERIMENTS = {  # name: (shuffle, snapshot, held-out session)
    "sh1 baseline (200 ep)": (1, 200, "110535"),
    "sh2 hist_eq": (2, 100, "110535"),
    "sh3 hist_eq+flip": (3, 100, "110535"),
    "sh4 baseline": (4, 100, "135059"),
    "sh5 hist_eq+flip": (5, 100, "135059"),
    "sh6 baseline": (6, 100, "171101"),
    "sh7 hist_eq+flip": (7, 100, "171101"),
}

runs = {}
for name, (shuffle, snapshot, test_session) in EXPERIMENTS.items():
    predictions = load_predictions(find_predictions(PROJECT, shuffle, snapshot))
    runs[name] = evaluate_run(predictions, labels, [test_session])

comparison = compare_runs(runs)
comparison.round(3)

C:\Users\edoua\AppData\Local\Temp\ipykernel_7352\1611084541.py:4: UserWarning: Session 150130 has no CollectedData_dani.h5, skipped.
  labels = load_labels(Path("data") / "labeled-data")


,test_sessions,keypoints,mean_error_px,median_error_px,mean_error_norm,pck,detection_rate,detected_error_px,confident_wrong_rate,false_detection_rate,orientation_accuracy
sh1 baseline (200 ep),110535,312,238.405,167.331,1.571,0.189,0.000,NaN,NaN,NaN,0.231
sh2 hist_eq,110535,312,86.025,24.112,0.567,0.353,0.080,21.909,0.200,NaN,0.051
sh3 hist_eq+flip,110535,312,21.109,12.530,0.139,0.615,0.359,9.539,0.000,NaN,0.974
sh4 baseline,135059,320,61.534,40.476,0.397,0.444,0.894,63.865,0.535,NaN,0.000
sh5 hist_eq+flip,135059,320,60.225,36.356,0.389,0.488,0.975,58.007,0.487,NaN,0.000
sh6 baseline,171101,160,15.260,4.151,0.154,0.881,0.588,7.464,0.032,NaN,0.900
sh7 hist_eq+flip,171101,160,9.559,3.429,0.097,0.925,0.712,4.749,0.000,NaN,1.000
